In [2]:
import pandas as pd
import great_expectations as gx

In [ ]:
context = gx.get_context(mode="ephemeral")
print(type(context))
print(context)

<class 'great_expectations.data_context.data_context.ephemeral_data_context.EphemeralDataContext'>
{
  "checkpoint_store_name": "checkpoint_store",
  "config_version": 4,
  "data_docs_sites": {
    "local_site": {
      "class_name": "SiteBuilder",
      "show_how_to_buttons": true,
      "store_backend": {
        "class_name": "TupleFilesystemStoreBackend",
        "base_directory": "C:\\Users\\migue\\AppData\\Local\\Temp\\tmp35mvmz3z"
      },
      "site_index_builder": {
        "class_name": "DefaultSiteIndexBuilder"
      }
    }
  },
  "expectations_store_name": "expectations_store",
  "fluent_datasources": {},
  "stores": {
    "expectations_store": {
      "class_name": "ExpectationsStore",
      "store_backend": {
        "class_name": "InMemoryStoreBackend"
      }
    },
    "validation_results_store": {
      "class_name": "ValidationResultsStore",
      "store_backend": {
        "class_name": "InMemoryStoreBackend"
      }
    },
    "checkpoint_store": {
      "class_n

In [4]:
model_input = pd.read_parquet("../data/02_intermediate/model_input_data.parquet")
data_source = context.data_sources.add_pandas(name="mortgage_datasource")
data_asset = data_source.add_dataframe_asset(name="model_input_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")

print(f"Linhas: {model_input.shape[0]}")
print(f"Colunas: {model_input.shape[1]}")
print(model_input.dtypes)

Linhas: 50000
Colunas: 33
credit_score                                   int64
first_payment_date                             int64
first_time_homebuyer_flag                        str
maturity_date                                  int64
msa                                          float64
mi_percentage                                  int64
number_of_units                                int64
occupancy_status                                 str
original_cltv                                  int64
original_dti                                   int64
original_upb                                   int64
original_ltv                                   int64
original_interest_rate                       float64
channel                                          str
prepayment_penalty_flag                          str
amortization_type                                str
property_state                                   str
property_type                                    str
postal_code         

In [5]:
batch = batch_definition.get_batch(batch_parameters={"dataframe": model_input})
print(type(batch))

<class 'great_expectations.datasource.fluent.interfaces.Batch'>


In [8]:
suite = context.suites.add(gx.ExpectationSuite(name="mortgage_quality_suite"))

# --- Estrutura ---
suite.add_expectation(gx.expectations.ExpectTableColumnCountToEqual(value=33))
suite.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=40000, max_value=60000))
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="loan_sequence_number"))
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="default"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="loan_sequence_number"))

# --- default ---
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="default", value_set=[0, 1]
))

# --- credit_score ---
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column="credit_score",
    value_set=list(range(300, 851)) + [9999]
))

# --- original_ltv ---
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="original_ltv", min_value=6, max_value=999
))

# --- original_interest_rate ---
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="original_interest_rate", min_value=0, max_value=30, strict_min=True
))

# --- msa null rate ---
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(
    column="msa", mostly=0.80
))

print(f"Expectativas definidas: {len(suite.expectations)}")

DataContextError: Cannot add ExpectationSuite with name mortgage_quality_suite because it already exists.

In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="mortgage_validation",
        data=batch_definition,
        suite=suite,
    )
)

results = validation_definition.run(batch_parameters={"dataframe": model_input})

Calculating Metrics:   0%|          | 0/47 [00:00<?, ?it/s]

Sucesso global: True

✅ expect_table_column_count_to_equal
✅ expect_table_row_count_to_be_between
✅ expect_column_to_exist
✅ expect_column_values_to_be_unique
✅ expect_column_to_exist
✅ expect_column_values_to_be_in_set
✅ expect_column_values_to_be_in_set
✅ expect_column_values_to_be_between
✅ expect_column_values_to_be_between
✅ expect_column_values_to_not_be_null


In [10]:
# --- relação first_payment_date < maturity_date ---
suite.add_expectation(gx.expectations.ExpectColumnPairValuesAToBeGreaterThanB(
    column_A="maturity_date",
    column_B="first_payment_date",
))

# --- relação original_cltv >= original_ltv ---
suite.add_expectation(gx.expectations.ExpectColumnPairValuesAToBeGreaterThanB(
    column_A="original_cltv",
    column_B="original_ltv",
    or_equal=True,
))

# re-correr com as novas expectativas
results = validation_definition.run(batch_parameters={"dataframe": model_input})

Calculating Metrics:   0%|          | 0/59 [00:00<?, ?it/s]